In [17]:
import re 
import pdfplumber
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# 🤖 LLM
llm = ChatGroq(model="llama-3.1-8b-instant")

# -------------------------------
# 🧾 State
# -------------------------------
class GraphState(TypedDict):
    resume_text: str
    job_description: str
    score: int
    feedback: str
    improved_resume: str
    iteration: int
    confidence: str

# -------------------------------
# 📄 PDF Extraction
# -------------------------------
def extract_pdf_text(file_path):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

# -------------------------------
# 🟢 ATS Evaluation

def extract_score(text):
    match = re.search(r'\b(\d{1,3})\b', text)
    if match:
        return int(match.group(1))
    return 60


def evaluate_node(state: GraphState):
    print(f"\n🔵 Evaluating Resume (Iteration {state['iteration']})...")

    prompt = f"""
    Analyze this resume based on the job description.

    Resume:
    {state['resume_text']}

    Job Description:
    {state['job_description']}

    Give:
    1. ATS Score out of 100 (only number)
    2. 3 improvement suggestions
    """

    response = llm.invoke(prompt).content

    score = extract_score(response)

    print("👉 Score:", score)

    return {
        "score": score,          # ✅ IMPORTANT
        "feedback": response     # ✅ IMPORTANT
    }

# -------------------------------
# 🔁 Improvement Node
# -------------------------------
def improve_node(state: GraphState):
    print("🟡 Improving Resume...")

    prompt = f"""
    Improve this resume based on feedback and job description.

    Resume:
    {state['resume_text']}

    Job Description:
    {state['job_description']}

    Feedback:
    {state['feedback']}

    Add missing keywords and improve clarity.
    Keep it professional.
    """

    response = llm.invoke(prompt)

    return {
        "resume_text": response.content,
        "iteration": state["iteration"] + 1
    }

# -------------------------------
# 🔀 Decision Logic
# -------------------------------
def decision_node(state: GraphState):
    if state["score"] >= 90:
        return "end"
    elif state["iteration"] >= 3:
        return "end"
    else:
        return "improve"

# -------------------------------
# 🟣 Confidence Score
# -------------------------------
def confidence_node(state: GraphState):
    print("🟣 Generating Confidence Score...")

    if state["score"] >= 90:
        confidence = "High chance of selection"
    elif state["score"] >= 75:
        confidence = "Moderate chance"
    else:
        confidence = "Low chance"

    return {"confidence": confidence}

# -------------------------------
# 🧩 Build Graph
# -------------------------------
graph = StateGraph(GraphState)

graph.add_node("evaluate", evaluate_node)
graph.add_node("improve", improve_node)
graph.add_node("confidence", confidence_node)

graph.set_entry_point("evaluate")

graph.add_conditional_edges(
    "evaluate",
    decision_node,
    {
        "end": "confidence",
        "improve": "improve"
    }
)

graph.add_edge("improve", "evaluate")
graph.add_edge("confidence", END)

app = graph.compile()

# -------------------------------
# ▶️ RUN
# -------------------------------
resume_path = "resume.pdf"

job_description = """
Looking for a Data Scientist with Python, Machine Learning,
Data Analysis, Pandas, and SQL skills.
"""

resume_path = r"C:\Users\Classic\Desktop\langGraph\myenv\KESHAV YADAV (3).pdf"
resume_text = extract_pdf_text(resume_path)


result = app.invoke({
    "resume_text": resume_text,
    "job_description": "Looking for a Data Scientist...",
    "iteration": 0,
    "score": 0,          # ✅ ADD THIS
    "feedback": ""       # ✅ ADD THIS
})

print("\n📄 FINAL RESUME:\n")
print(result["resume_text"])

print("\n⭐ FINAL SCORE:", result["score"])
print("\n📊 CONFIDENCE:", result["confidence"])


🔵 Evaluating Resume (Iteration 0)...
👉 Score: 72
🟡 Improving Resume...

🔵 Evaluating Resume (Iteration 1)...
👉 Score: 92
🟣 Generating Confidence Score...

📄 FINAL RESUME:

**Improved Resume:**

**YOGESH**
yogesh-tanwar-2589b92a5/ | LinkedIn Email: yogeshtanwar99108@gmail.com
yogeshsr72 | github.com Mobile: +91 8059972208

**SUMMARY**
Highly motivated and detail-oriented Data Scientist with a Master's degree in Computer Application (MCA) and experience in developing machine learning models and data analysis projects. Proficient in Python, SQL, Java, and Machine Learning with a strong foundation in data science tools and technologies.

**EDUCATION**

* **Master of Computer Application (MCA)**, Central University of Haryana, Haryana, India (June 2024 – June 2026)
* **Post Graduation Diploma (PGDCA)**, Maharshi Dayanand University (MDU), Rohtak, Haryana, India (July 2022 – June 2023)
* **Bachelor of Science (B.sc)**, Maharshi Dayanand University (MDU), Rohtak, Haryana, India (July 2019 – 